In [1]:
import pandas as pd
from pathlib import Path

base_dir = Path("/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/APOLLO_Broad/Clinical_data")

patient_id_file = base_dir / "MBCAPOLLO-ApolloPatientid_DATA_2026-04-17_1022.csv"

treatment_files = [
    base_dir / "MBCAPOLLO-PalliativeTreatment_Beyond2_line_DATA_LABELS_2026-04-17_1547.csv",
    base_dir / "MBCAPOLLO-PalliativeTreatment1_2_line_DATA_LABELS_2026-04-17_1546.csv"
]

In [2]:
patient_map = pd.read_csv(patient_id_file, dtype=str)

patient_map.columns = patient_map.columns.str.strip()

patient_map["record_id"] = patient_map["record_id"].astype(str).str.strip()
patient_map["redcap_id"] = patient_map["redcap_id"].astype(str).str.strip()

patient_map = patient_map[
    patient_map["record_id"].notna() &
    (patient_map["record_id"] != "") &
    patient_map["redcap_id"].notna() &
    (patient_map["redcap_id"] != "")
]

id_to_redcap = dict(zip(patient_map["record_id"], patient_map["redcap_id"]))

print("Mapping file summary")
print("--------------------")
print("Unique record_id:", patient_map["record_id"].nunique())
print("Unique redcap_id:", patient_map["redcap_id"].nunique())

Mapping file summary
--------------------
Unique record_id: 335
Unique redcap_id: 327


In [3]:
summary = []

for infile in treatment_files:
    df = pd.read_csv(infile, dtype=str)
    df.columns = df.columns.str.strip()

    if "Patient ID" not in df.columns:
        raise ValueError(f"'Patient ID' column not found in {infile.name}")

    df["Patient ID"] = df["Patient ID"].astype(str).str.strip()

    # Remove old redcap_id column if present
    if "redcap_id" in df.columns:
        df = df.drop(columns=["redcap_id"])

    # Map redcap_id using Patient ID
    redcap_series = df["Patient ID"].map(id_to_redcap)

    # Insert redcap_id immediately after Patient ID
    patient_id_position = df.columns.get_loc("Patient ID")
    df.insert(patient_id_position + 1, "redcap_id", redcap_series)

    # Save updated file
    outfile = infile.with_name(infile.stem + "_with_redcap_id.csv")
    df.to_csv(outfile, index=False)

    # Count valid unique IDs
    valid_patient_ids = df["Patient ID"].replace("nan", pd.NA).dropna()
    valid_patient_ids = valid_patient_ids[valid_patient_ids != ""]

    valid_redcap_ids = df["redcap_id"].replace("nan", pd.NA).dropna()
    valid_redcap_ids = valid_redcap_ids[valid_redcap_ids != ""]

    unmatched = sorted(
        set(valid_patient_ids.unique()) -
        set(patient_map["record_id"].unique())
    )

    summary.append({
        "file": infile.name,
        "unique_patient_id": valid_patient_ids.nunique(),
        "unique_redcap_id": valid_redcap_ids.nunique(),
        "unmatched_patient_id_count": len(unmatched),
        "output_file": outfile.name
    })

    print(f"\nProcessed: {infile.name}")
    print("Unique Patient ID:", valid_patient_ids.nunique())
    print("Unique redcap_id:", valid_redcap_ids.nunique())
    print("Unmatched Patient IDs:", len(unmatched))

    if unmatched:
        print("Unmatched IDs:")
        print(unmatched)

    print("Saved:", outfile)


Processed: MBCAPOLLO-PalliativeTreatment_Beyond2_line_DATA_LABELS_2026-04-17_1547.csv
Unique Patient ID: 337
Unique redcap_id: 326
Unmatched Patient IDs: 2
Unmatched IDs:
['336', '337']
Saved: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/APOLLO_Broad/Clinical_data/MBCAPOLLO-PalliativeTreatment_Beyond2_line_DATA_LABELS_2026-04-17_1547_with_redcap_id.csv

Processed: MBCAPOLLO-PalliativeTreatment1_2_line_DATA_LABELS_2026-04-17_1546.csv
Unique Patient ID: 337
Unique redcap_id: 326
Unmatched Patient IDs: 2
Unmatched IDs:
['336', '337']
Saved: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/APOLLO_Broad/Clinical_data/MBCAPOLLO-PalliativeTreatment1_2_line_DATA_LABELS_2026-04-17_1546_with_redcap_id.csv


In [4]:
summary_df = pd.DataFrame(summary)
summary_df

,file,unique_patient_id,unique_redcap_id,unmatched_patient_id_count,output_file
0,MBCAPOLLO-PalliativeTreatment_Beyond2_line_DAT...,337,326,2,MBCAPOLLO-PalliativeTreatment_Beyond2_line_DAT...
1,MBCAPOLLO-PalliativeTreatment1_2_line_DATA_LAB...,337,326,2,MBCAPOLLO-PalliativeTreatment1_2_line_DATA_LAB...
